# Data Collection — Popular Languages via Web Scraping

**Module 1 · Step 2 of 2**

Scrapes an HTML table from IBM Skills Network to collect programming language names  
and their average annual salaries.

**Output:** `outputs/popular-languages.csv`

---
## 1. Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "beautifulsoup4"], check=True)

import requests
import csv
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

SOURCE_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud"
    "/IBM-DA0321EN-SkillsNetwork/labs/datasets/Programming_Languages.html"
)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "popular-languages.csv"

print("Setup complete.")

---
## 2. Download & Parse HTML

In [ ]:
response = requests.get(SOURCE_URL)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
table = soup.find("table")

print(f"Status: {response.status_code}")
print(f"Table found: {table is not None}")

---
## 3. Extract Language Name and Salary

In [ ]:
# Column layout: [rank, language, created_by, salary, learning_difficulty]
popular_languages = []

for row in table.find_all("tr"):
    cols = row.find_all("td")
    if len(cols) >= 4:
        language = cols[1].get_text(strip=True)
        salary   = cols[3].get_text(strip=True)
        if language.lower() == "language":  # skip header row that uses <td>
            continue
        popular_languages.append((language, salary))

print(f"Languages scraped: {len(popular_languages)}")
popular_languages

---
## 4. Save to CSV

In [ ]:
with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Language", "Average Annual Salary"])
    writer.writerows(popular_languages)

print(f"Saved: {OUTPUT_FILE}")

---
## 5. Preview & Findings

In [ ]:
df = pd.read_csv(OUTPUT_FILE)
print(f"Rows: {len(df)}")
df